In [1]:
import sys
from pathlib import Path


def get_project_root(project_dir_name="Project", marker=".git"):
    """
    1) Walk upwards until we find the repo root (contains .git).
    2) Return <repo_root>/<project_dir_name> as the actual project root
       (the folder that contains models/, training/, notebooks/, etc.).
    3) Add that project root to sys.path for imports.
    """
    current = Path.cwd().resolve()

    # Step 1: find repo root by .git
    repo_root = None
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            repo_root = parent
            break

    if repo_root is None:
        raise RuntimeError(f"Repo root not found (no '{marker}' directory)")

    # Step 2: define actual project root
    project_root = repo_root / project_dir_name
    if not project_root.exists():
        raise RuntimeError(
            f"Found repo root at {repo_root}, but '{project_dir_name}' folder not found."
        )

    # Step 3: add to sys.path
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    print(f"[OK] Repo root:    {repo_root}")
    print(f"[OK] Project root: {project_root}")
    return project_root

PROJECT_ROOT = get_project_root()

[OK] Repo root:    /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning
[OK] Project root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project


In [2]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Evaluating on:", device)

Evaluating on: mps


In [3]:
from pathlib import Path
import json
import torch


SEQ_DIR = PROJECT_ROOT / "03_Sequences"
RUN_DIR = PROJECT_ROOT / "runs" / "LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC"  # <- pick one


CONFIG_PATH = RUN_DIR / "config.json"
CKPT_PATH   = RUN_DIR / "best_model.pt"


In [4]:
from utils.data_utils import make_test_loader

test_loader, X_test_raw, y_test, config = make_test_loader(
    run_dir=RUN_DIR,
    seq_dir=SEQ_DIR,
    shuffle=False
)

model_class_name = config["model_class"]
model_kwargs     = config["model_kwargs"]
seq_len = config["seq_len"]
SEQ_LEAF_DIR = SEQ_DIR / f"seq{seq_len}"   # SEQ_DIR is base: .../03_Sequences

In [5]:
from models import LSTMClassifier  # add more as you create them
import hashlib

MODEL_REGISTRY = {
    "LSTMClassifier": LSTMClassifier,
    # "GRUClassifier": GRUClassifier,
    # "TransformerClassifier": TransformerClassifier,
}

ModelClass = MODEL_REGISTRY[model_class_name]
best_model = ModelClass(**model_kwargs).to(device)
best_model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
best_model.eval()

def fp(m):
    h = hashlib.sha256()
    for k in sorted(m.state_dict().keys()):
        t = m.state_dict()[k].detach().cpu().contiguous().numpy()
        h.update(k.encode()); h.update(t.tobytes())
    return h.hexdigest()[:16]

print("MODEL_FP:", fp(best_model))

print("Model loaded from:", CKPT_PATH)


MODEL_FP: 6eae0c985503c3f6
Model loaded from: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC/best_model.pt


In [6]:
import numpy as np
from tqdm import tqdm

all_probs  = []
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch_i, (X_batch, y_batch) in enumerate(tqdm(test_loader, desc="Predicting (test)", leave=False)):
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1).to(device)

        logits = best_model(X_batch).view(-1)                 # ensure shape (batch,)
        probs  = torch.sigmoid(logits)                   # UP probability in [0,1]
        preds  = (probs >= 0.5).long()

        ########################################
        # DEBUG: first batch only
        if batch_i == 0:
            # 1) Model fingerprint (again, right now)
            import hashlib, torch
            def fp(m):
                h = hashlib.sha256()
                for k in sorted(m.state_dict().keys()):
                    t = m.state_dict()[k].detach().cpu().contiguous().numpy()
                    h.update(k.encode()); h.update(t.tobytes())
                return h.hexdigest()[:16]

            print("MODEL_FP:", fp(best_model))

            # 2) Input fingerprint (whole tensor, not just 5 values)
            xb = X_batch.detach().cpu().contiguous().numpy()
            print("X_FP:", hashlib.sha256(xb.tobytes()).hexdigest()[:16])

            # 3) Logits fingerprint
            lg = logits.detach().cpu().contiguous().numpy()
            print("LOGITS_FP:", hashlib.sha256(lg.tobytes()).hexdigest()[:16])

            # 4) Preds fingerprint
            pr = preds.detach().cpu().contiguous().numpy()
            print("PREDS_FP:", hashlib.sha256(pr.tobytes()).hexdigest()[:16])
            
            print("model.training:", best_model.training)
            print("param dtype:", next(best_model.parameters()).dtype)
            print("X dtype:", X_batch.dtype)
            print("device:", next(best_model.parameters()).device)
        ########################################

        all_probs.extend(probs.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

all_probs  = np.array(all_probs).reshape(-1)
all_preds  = np.array(all_preds).reshape(-1).astype(int)
all_labels = np.array(all_labels).reshape(-1).astype(int)


Predicting (test):   0%|          | 27/29592 [00:00<12:54, 38.15it/s] 

MODEL_FP: 6eae0c985503c3f6
X_FP: bafdebb3d1c993b8
LOGITS_FP: 8a4b4452a9b2f74a
PREDS_FP: 76c862828aa3b269
model.training: False
param dtype: torch.float32
X dtype: torch.float32
device: mps:0


KeyboardInterrupt: 

In [ ]:
# -------------------------------------------------------------
# 2) Accuracy + confusion per t_to_end_min
# -------------------------------------------------------------
# Here we go beyond global metrics and analyze performance
# as a function of "minutes to end of 15-min window".
#
# Steps:
#   1) Find where t_to_end_min lives in the feature vector
#   2) Extract t_to_end_min for each TEST sample
#      (from the *unscaled* X_test, last time step in each sequence)
#   3) Run a forward pass over test_loader to collect:
#         - predicted labels
#         - true labels
#   4) Build a DataFrame and compute:
#         - TP/FP/TN/FN per t_to_end_min
#         - accuracy per t_to_end_min
# -------------------------------------------------------------
import pandas as pd
from tqdm.auto import tqdm

# 2.1) Find the feature index for t_to_end_min
t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
print(f"t_to_end_min is at feature index: {t_to_end_min_idx}")

# 2.2) Extract t_to_end_min from the UN-SCALED test data
#      Each sequence has shape (seq_len, num_features).
#      We take the LAST timestep [-1] for each sample:
#         → this corresponds to the "current" minute the model is predicting for.
t_to_end_min_values = X_test_raw[:, -1, t_to_end_min_idx]  # shape: (num_test,)

print(f"Unique t_to_end_min values: {np.unique(t_to_end_min_values)}")

# 2.3) Collect predictions and true labels from the best model
best_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_i, (X_batch, y_batch) in enumerate(
        tqdm(test_loader, desc="Predicting (test)", leave=False)
    ):
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1).to(device)

        logits = best_model(X_batch).view(-1)     # IMPORTANT: use best_model
        probs  = torch.sigmoid(logits)
        preds  = (probs >= 0.5).long()

        if batch_i == 0:
            xb = X_batch.detach().cpu().contiguous().numpy()
            lg = logits.detach().cpu().contiguous().numpy()
            pr = preds.detach().cpu().contiguous().numpy()

            def fp(m):
                h = hashlib.sha256()
                for k in sorted(m.state_dict().keys()):
                    t = m.state_dict()[k].detach().cpu().contiguous().numpy()
                    h.update(k.encode()); h.update(t.tobytes())
                return h.hexdigest()[:16]

            print("MODEL_FP:", fp(best_model))
            print("X_FP:", hashlib.sha256(xb.tobytes()).hexdigest()[:16])
            print("LOGITS_FP:", hashlib.sha256(lg.tobytes()).hexdigest()[:16])
            print("PREDS_FP:", hashlib.sha256(pr.tobytes()).hexdigest()[:16])

            print("training:", best_model.training)
            print("param dtype:", next(best_model.parameters()).dtype)
            print("X dtype:", X_batch.dtype)
            print("device:", next(best_model.parameters()).device)

        # IMPORTANT: extend with numpy arrays, not torch tensors
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

all_preds  = np.array(all_preds, dtype=int).reshape(-1)
all_labels = np.array(all_labels, dtype=int).reshape(-1)

assert (
    all_preds.shape[0] == t_to_end_min_values.shape[0]
), "Mismatch: number of test predictions != number of t_to_end_min entries"


# 2.4) Build a DataFrame for per-minute analysis
df_results = pd.DataFrame(
    {
        "t_to_end_min": t_to_end_min_values,
        "y_true": all_labels,
        "y_pred": all_preds,
    }
)

# Add confusion components per sample
df_results["tp"] = ((df_results.y_true == 1) & (df_results.y_pred == 1)).astype(int)
df_results["fp"] = ((df_results.y_true == 0) & (df_results.y_pred == 1)).astype(int)
df_results["tn"] = ((df_results.y_true == 0) & (df_results.y_pred == 0)).astype(int)
df_results["fn"] = ((df_results.y_true == 1) & (df_results.y_pred == 0)).astype(int)
df_results["correct"] = (df_results["tp"] + df_results["tn"]).astype(int)

# 2.5) Aggregate stats by t_to_end_min
stats = (
    df_results.groupby("t_to_end_min")
    .agg(
        count=("correct", "count"),
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        tn=("tn", "sum"),
        fn=("fn", "sum"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
)

stats["accuracy_pct"] = stats["accuracy"] * 100

# 2.6) Print per-minute results
print("\n" + "=" * 70)
print("ACCURACY + CONFUSION METRICS PER t_to_end_min")
print("=" * 70)
print(stats.to_string(index=False))
print("=" * 70)

# 2.7) Baseline vs model
# Baseline (always predicting 1) accuracy = fraction of positives in test labels
baseline_acc = all_labels.mean()
model_acc = (all_preds == all_labels).mean()

print(f"\nBaseline Accuracy (always predict 1): {baseline_acc:.4f}")
print(f"Model Test Accuracy (from preds):      {model_acc:.4f}")

In [ ]:
def hash_array(a: np.ndarray) -> str:
    return hashlib.md5(a.tobytes()).hexdigest()

print("X_test shape:", X_test_raw.shape)
print("y_test shape:", y_test.shape)
print("X_test hash:", hash_array(X_test_raw))
print("y_test hash:", hash_array(y_test))